# SoftBank Investment Analysis Agent
## Multi-Agent AI System for Startup Due Diligence

**For Students** — Learn how to build AI systems that analyze startup pitches automatically.

In this notebook, you'll:
- Build 4 specialized AI agents
- Chain them together to work sequentially
- Analyze a real startup pitch
- Get a professional investment memo

**Time to run:** 2-5 minutes per startup | **Time saved vs manual:** 4-8 hours

## 📚 Learning Objectives

By the end of this notebook, you'll understand:

✅ **How multi-agent systems work** — Multiple AI agents specializing in different tasks

✅ **Context chaining** — How agents see and use previous agents' outputs

✅ **Prompt engineering** — Writing effective instructions for AI agents

✅ **Real-world AI application** — Automating complex business processes

✅ **Tool integration** — Connecting AI to external tools (web search)

✅ **Source attribution** — Preventing AI hallucination through citations

## Section 1: Setup & Environment

### WHAT are we doing?
Installing dependencies and configuring the environment so we can use CrewAI, Gemini, and Tavily.

### WHY does this matter?
We need the right tools:
- **CrewAI** = orchestration framework (makes agents work together)
- **Gemini 3.5 Flash-Lite** (Google) = our AI brain (reasoning & analysis)
- **Tavily** = web search tool (real-time data)

### OBSERVE
After running this cell, you should see: ✅ Setup complete

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from .env file
load_dotenv()

# Import CrewAI components
from crewai import Agent, Task, Crew, Process
from crewai_tools import TavilySearchTool

# Set up API keys
# In production: use environment variables
# For testing: set them directly here
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY", "your-key-here")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY", "your-key-here")

# Initialize the web search tool
search_tool = TavilySearchTool()

print("✅ Setup complete")
print(f"✅ GEMINI_API_KEY ready: {'GEMINI_API_KEY' in os.environ}")
print(f"✅ TAVILY_API_KEY ready: {'TAVILY_API_KEY' in os.environ}")

## Section 2: Understanding the Architecture

### WHAT is our system doing?
```
Pitch Deck
    ↓
Agent 1: Extract Claims
    ↓ (output: claims list)
Agent 2: Research Startup (uses Tavily)
    ↓ (output: research findings)
Agent 3: Verify Claims
    ↓ (output: risk matrix)
Agent 4: Write Memo
    ↓
Investment Memo (Ready for Committee)
```

### WHY this order?
- **Extract first** — Agent 2 knows which claims to research
- **Research second** — Agent 3 can compare claims to findings
- **Verify third** — Agent 4 has all the info to write a complete memo
- **Synthesize last** — Final memo includes everything with sources

### OBSERVE
Each agent passes information to the next (context chaining). Later agents know what earlier agents found.

## Section 3: Configure the Shared LLM

### WHAT is Gemini 3.5 Flash-Lite?
Gemini 3.5 Flash-Lite is an AI model from Google. It's:
- Smart enough for complex analysis
- Fast (good for real-time use)
- Free to start: a Google AI Studio key (aistudio.google.com) needs no credit card; the free tier has rate limits

### WHY do all agents use the same LLM?
- **Consistency** — Same model, same thinking style
- **Cost efficiency** — One configuration, not 4
- **Easy to modify** — Change model once, applies everywhere

Example: Want a bigger model? Try `gemini/gemini-2.5-flash`. Change it in one place: the `MODEL` line below, or a `GEMINI_MODEL` entry in your .env.

Free keys get about 20 requests per model per day. One run here uses around 10, because the Research Agent is capped at 5 searches.

### OBSERVE
The free tier sometimes replies "busy" (errors 429 / 503). The `retry` settings make the client wait and try again, so a busy minute doesn't crash the whole pipeline.

We leave `temperature` at the model's default.
Our agents stay grounded through their **instructions**: cite every claim, and say "no data found" rather than guess.

In [ ]:
from crewai import LLM
from google.genai import types

# The free tier sometimes answers "busy" (429 / 503): wait and retry instead of crashing
retry = types.HttpRetryOptions(attempts=5, initial_delay=5, max_delay=60, http_status_codes=[429, 500, 503])

# Free keys get ~20 requests per model per day. Out of quota? Pick another model:
# each one has its own daily allowance (e.g. gemini/gemini-2.5-flash, gemini/gemini-3.1-flash-lite)
MODEL = os.getenv("GEMINI_MODEL", "gemini/gemini-3.5-flash-lite")

# Shared LLM for all agents (CrewAI talks to Gemini natively; reads GEMINI_API_KEY)
llm = LLM(
    model=MODEL,                      # Google Gemini 3.5 Flash-Lite by default
    max_tokens=8192,                  # room for a full memo
    client_params={"http_options": types.HttpOptions(retry_options=retry)},
)

print("✅ LLM Configuration:")
print(f"   Model: {llm.model}")
print(f"   Max tokens: 8192")
print("\n💡 Tip: All agents will use this exact LLM")

## Section 4: Agent 1 — Claims Analyst

### WHAT does this agent do?
Reads a startup pitch deck and extracts ALL factual claims:
- Revenue ($120K ARR)
- Growth rates (235% YoY)
- Team backgrounds (CEO ex-Google)
- Market sizes ($8B TAM)
- Traction (25 customers)

### WHY is extraction the first step?
We need to know WHAT to research. If the pitch says "$4.2M ARR", Agent 2 will search for that specific claim.

### OBSERVE
The backstory ("You are a meticulous analyst...") helps Gemini think like an investment pro, not a generic AI.

### ❓ Q&A
**Q: Why no tools for Agent 1?**
A: It just reads text. No need to search the web.

**Q: How does it know which lines are claims?**
A: We tell it in the goal. Gemini figures out the rest.

In [ ]:
# AGENT 1: Claims Analyst
claims_analyst = Agent(
    role="Claims Analyst",
    
    goal=(
        "Extract ALL factual claims from the pitch deck. Include: "
        "revenue, growth rates, market sizes, team backgrounds, "
        "customer metrics, and product specifications. "
        "Record the EXACT source line for each claim."
    ),
    
    backstory=(
        "You are a meticulous investment analyst with 15+ years "
        "of experience reading pitch decks. You extract claims with "
        "surgical precision and always record the source line."
    ),
    
    verbose=True,                          # Show reasoning in console
    allow_delegation=False,                # Work independently
    llm=llm,                               # Use shared Gemini LLM
)

print("✅ Agent 1 Created: Claims Analyst")
print(f"   Role: {claims_analyst.role}")
print(f"   No tools needed (direct text analysis)")
print("\n💡 This agent is like a detective scanning the pitch for key facts.")

## Section 5: Agent 2 — Research Agent

### WHAT does this agent do?
Searches the web for information about the startup:
- News and press releases
- Founder backgrounds (LinkedIn)
- Funding history
- Customer references
- Red flags or controversies

### WHY use Tavily (web search)?
We need REAL data, not imagination. If the pitch says "raised $15M from a16z", we verify it's true.

### OBSERVE
This is the SLOWEST agent (45 seconds) because it makes real web API calls.

### ❓ Q&A
**Q: What if Tavily finds nothing?**
A: Agent explicitly says "No data found". That's important info!

**Q: Does this agent see Agent 1's output?**
A: Yes! Via `context=[task1]`. It knows which claims to research.

In [ ]:
# AGENT 2: Research Agent (has web search tool)
research_agent = Agent(
    role="Research Agent",
    
    goal=(
        "Search the web for public information about this startup. "
        "Find: news, funding records, founder backgrounds, "
        "competitors, and market data. Return all findings with "
        "EXACT URLs for citation."
    ),
    
    backstory=(
        "You are a thorough research analyst. You search multiple "
        "sources and document everything. You never guess or make "
        "up data. If something isn't on the web, you say so."
    ),
    
    tools=[search_tool],                   # ← This agent has a tool!
    max_iter=6,                            # ← at most ~5 searches: keeps a run inside the free quota
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

print("✅ Agent 2 Created: Research Agent")
print(f"   Role: {research_agent.role}")
print(f"   Tool: Tavily (web search)")
print("\n💡 This agent is like a journalist fact-checking the pitch.")

## Section 6: Agent 3 — Verification Agent

### WHAT does this agent do?
Compares each claim from Agent 1 against the research from Agent 2:
- ✅ **VERIFIED** — Claim matches public data
- ⚠️ **CONTRADICTED** — Claim conflicts with data (RED FLAG)
- ❓ **UNVERIFIED** — No public data found (needs human review)

### WHY do we need this step?
Some startups exaggerate. We catch inconsistencies.

Example:
- Pitch says: "Only competitor in this space"
- Research finds: 5 competitors already exist
- Agent 3 flags: "CONTRADICTED (High confidence)"

### OBSERVE
This is where we assign **confidence levels**:
- 🔴 HIGH = clear evidence of problem
- 🟠 MEDIUM = some evidence
- 🟡 LOW = minor issue

### ❓ Q&A
**Q: Can this agent see both Agent 1 AND Agent 2 outputs?**
A: YES! That's the power of context chaining: `context=[task1, task2]`

**Q: What's a "red flag"?**
A: A contradiction that suggests the founder is lying or confused.

In [ ]:
# AGENT 3: Verification Agent (compares claims to research)
verifier = Agent(
    role="Verification Agent",
    
    goal=(
        "Compare each claim from the pitch against research findings. "
        "Mark as: VERIFIED, CONTRADICTED, or UNVERIFIED. "
        "Assign confidence levels (High/Medium/Low) to red flags. "
        "Flag anything that doesn't match public data."
    ),
    
    backstory=(
        "You are a forensic analyst skilled at spotting red flags. "
        "You distinguish between: verified (good), contradicted (bad), "
        "and unverified (needs checking). You document every finding."
    ),
    
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

print("✅ Agent 3 Created: Verification Agent")
print(f"   Role: {verifier.role}")
print(f"   No tools (compares information)")
print("\n💡 This agent is like a fact-checker marking claims as TRUE/FALSE/UNKNOWN.")

## Section 7: Agent 4 — Memo Writer

### WHAT does this agent do?
Synthesizes ALL findings into a professional investment memo:
- Executive summary (rating + recommendation)
- Company overview
- Investment thesis (why invest?)
- Key strengths (verified claims)
- Red flags (with confidence levels)
- Unverified claims (needs human follow-up)
- Final recommendation (INVEST / PASS / INVESTIGATE)

### WHY is this the final step?
By now, we have:
- ALL claims extracted (Agent 1)
- Research on each claim (Agent 2)
- Verification status (Agent 3)

Agent 4 just needs to organize and present it professionally.

### OBSERVE
**CRITICAL:** Every single claim must be cited:
- From pitch deck: (Pitch: "Sales & Marketing section")
- From web: (Source: https://techcrunch.com/...)

NO citation = NO claim. This prevents hallucination.

### ❓ Q&A
**Q: Does Agent 4 know what all other agents found?**
A: YES! `context=[task1, task2, task3]` gives it everything.

**Q: What makes a good memo?**
A: Clear structure + source citations + realistic recommendations.

In [ ]:
# AGENT 4: Memo Writer (synthesizes everything)
memo_writer = Agent(
    role="Investment Memo Writer",
    
    goal=(
        "Write a professional investment memo synthesizing ALL findings. "
        "Include: executive summary, company overview, thesis, strengths, "
        "red flags, unverified claims, and final recommendation. "
        "EVERY CLAIM MUST CITE ITS SOURCE (pitch line or URL)."
    ),
    
    backstory=(
        "You are a seasoned investment analyst who has written "
        "hundreds of memos. You present complex information clearly "
        "with source citations. Your memos are read by senior partners "
        "making million-dollar decisions."
    ),
    
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

print("✅ Agent 4 Created: Memo Writer")
print(f"   Role: {memo_writer.role}")
print(f"   No tools (synthesis + writing)")
print("\n💡 This agent is like the final report writer who pulls everything together.")

## Section 8: Define Tasks (Work for Each Agent)

### WHAT is a Task?
A Task tells an agent:
1. **What to do** (description)
2. **What success looks like** (expected_output)
3. **Who does it** (agent)
4. **What context they have** (prior agent outputs)

### WHY do we separate agents from tasks?
Agent = personality (role, backstory, tools)
Task = specific job (extract claims from THIS pitch deck)

One agent can do multiple tasks. One task needs one agent.

### OBSERVE
Notice the `context` parameter:
- Task 1: No context (reads the pitch)
- Task 2: `context=[task1]` (sees Agent 1's claims)
- Task 3: `context=[task1, task2]` (sees Agents 1 & 2)
- Task 4: `context=[task1, task2, task3]` (sees everyone)

This is **context chaining** in action!

In [ ]:
# TASK 1: Extract Claims
task1 = Task(
    description=(
        "Read the pitch deck below. First write down the company name exactly as it appears. "
        "Then extract ALL factual claims: revenue, pricing, customers, growth rates, "
        "market size, team experience, traction, projections, and partnerships. "
        "Quote each claim with the pitch section it came from.\n\n"
        "PITCH DECK:\n{pitch_deck_text}"
    ),
    expected_output="Company name, then a numbered list of claims, each with its pitch section",
    agent=claims_analyst,
)

# TASK 2: Research (sees Task 1 output)
task2 = Task(
    description=(
        "Search the web for public information about this startup: news, funding history, "
        "founders' backgrounds, competitors, and market reports. "
        "Always include the company name in searches about the company or its people "
        "(e.g. '<company> <founder name>'); never search a person's name on its own. "
        "Use at most 5 searches, so pick them well: the company, its founders, its funding, "
        "the market size, and its competitors. "
        "For every finding, give the full source URL. If a search finds nothing, say so."
    ),
    expected_output="Research findings, each with its full source URL",
    agent=research_agent,
    context=[task1],  # ← This agent sees Task 1's output
)

# TASK 3: Verify (sees Tasks 1 & 2)
task3 = Task(
    description=(
        "Step 1: check the pitch against ITSELF. Do the numbers agree with each other? "
        "(price × number of customers vs. stated ARR, LTV vs. price, projections vs. current traction). "
        "Show the arithmetic. Any internal mismatch is CONTRADICTED.\n"
        "Step 2: compare each claim with the research findings. "
        "Mark every claim VERIFIED, CONTRADICTED, or UNVERIFIED, and give each red flag "
        "a confidence level (High / Medium / Low) with a one-line reason."
    ),
    expected_output="Verification table: claim, verdict, confidence, evidence (arithmetic or URL)",
    agent=verifier,
    context=[task1, task2],  # ← This agent sees Tasks 1 & 2
)

# TASK 4: Write Memo (sees all Tasks 1, 2, 3)
task4 = Task(
    description=(
        "Write an investment memo titled with the company's name: executive summary, company "
        "overview, thesis, strengths, red flags (with confidence), unverified items, "
        "and a recommendation (INVEST / PASS / INVESTIGATE) with next diligence steps. "
        "Cite everything: pitch claims as (Pitch: section), web facts with their full URL. "
        "Put the arithmetic behind any numeric red flag in the memo. Do not add a date."
    ),
    expected_output="Professional investment memo (Markdown) ready for committee, with URLs",
    agent=memo_writer,
    output_file="reports/investment_memo.md",  # ← memo saved to disk
    context=[task1, task2, task3],  # ← This agent sees all prior tasks
)

print("✅ All 4 Tasks Created")
print("   Task 1: Extract (no context)")
print("   Task 2: Research (sees Task 1)")
print("   Task 3: Verify (sees Tasks 1 & 2)")
print("   Task 4: Memo (sees Tasks 1, 2, & 3)")
print("\n💡 Context flows forward: each task knows what earlier tasks found.")

## Section 9: Load Sample Pitch Deck

### WHAT are we analyzing?
Horizon AI Labs — a fictional startup raising a $5M Series A for supply-chain AI (`sample_data/sample_pitch_deck.txt`).

### WHY this example?
It's realistic, and it hides real problems:
- $15K/month × 8 pilots doesn't add up to $120K ARR
- "Proven unit economics" with zero paying non-pilot customers
- $120K → $2.5M ARR in one year

### OBSERVE
When you run the full pipeline, watch for:
- Which claims get VERIFIED (have supporting evidence)
- Which get CONTRADICTED (conflict with data)
- Which stay UNVERIFIED (no public data found)

### ❓ Q&A
**Q: Can I use my own pitch deck?**
A: YES! Put your pitch text in a file and point `PITCH_FILE` at it.

**Q: Does the pitch need to be formatted a certain way?**
A: No. Plain text works. PDF/images would need OCR first.

In [ ]:
# Load the sample pitch deck (fictional Horizon AI Labs)
PITCH_FILE = "sample_data/sample_pitch_deck.txt"

with open(PITCH_FILE, encoding="utf-8") as f:
    sample_pitch = f.read()

print("✅ Sample Pitch Loaded: Horizon AI Labs (Series A)")
print(f"   Length: {len(sample_pitch)} characters")
print("\n💡 Tip: When we run the pipeline, watch which claims get verified!")

## Section 10: Assemble the Crew

### WHAT is a Crew?
A Crew is:
- List of all agents
- List of all tasks (in order)
- Process type (how tasks run)
- Configuration (verbose mode, memory)

### WHY sequential process?
We need Agent 2 to finish before Agent 3 starts (Agent 3 needs Agent 2's output).

Parallel would be faster but wouldn't work for us.

### OBSERVE
- `verbose=True` shows agent reasoning in console (educational!)
- Context flows between tasks through `context=[...]` — no extra memory store needed

### ❓ Q&A
**Q: What does "sequential" mean?**
A: Task 1 → Task 2 → Task 3 → Task 4 (one after another, in order)

**Q: Can we run tasks in parallel?**
A: Yes, but only if they don't depend on each other. Here they do.

In [ ]:
# Create the Crew (all agents + tasks + configuration)
crew = Crew(
    agents=[claims_analyst, research_agent, verifier, memo_writer],
    tasks=[task1, task2, task3, task4],
    process=Process.sequential,  # Run one after another
    verbose=True,                # Show detailed reasoning
    max_rpm=8,                   # Stay under the Gemini free tier's per-minute limit
)

print("✅ Crew Assembled")
print("")
print("   Agents (4 total):")
print("   1. Claims Analyst (extract)")
print("   2. Research Agent (search web)")
print("   3. Verification Agent (cross-check)")
print("   4. Memo Writer (synthesize)")
print("")
print("   Process: Sequential (Task 1 → 2 → 3 → 4)")
print("   Verbose: ON (you'll see reasoning)")
print("   Context: chained with context=[...]")
print("\n💡 The crew is ready. Time to run the analysis!")

## Section 11: Run the Analysis Pipeline

### WHAT happens now?
All 4 agents work in sequence:
1. Agent 1 extracts ~30-50 claims from the pitch
2. Agent 2 searches the web for each claim (~7-10 searches)
3. Agent 3 compares claims vs findings
4. Agent 4 writes professional memo

### WHY does it take 2-5 minutes?
- Agent 1: ~15 sec (local)
- Agent 2: ~45 sec (web API calls)
- Agent 3: ~15 sec (local)
- Agent 4: ~15 sec (local)

Agent 2 is slow because it makes real web requests.

### OBSERVE
**Watch the console output:**
- "Agent 1 thinking..." → claims being extracted
- "Agent 2 searching..." → web queries happening
- "Agent 3 analyzing..." → comparisons being made
- "Agent 4 writing..." → memo being composed

### ❓ Q&A
**Q: Can I interrupt if it's taking too long?**
A: Yes, but wait at least 2 minutes (Agent 2 needs time).

**Q: What if an API fails?**
A: You'll see an error. Just re-run this cell.

In [ ]:
print("🚀 STARTING PIPELINE...")
print("="*60)
print("")
print("This will take 2-5 minutes.")
print("Agent 2 (web search) is the slow part. That's normal!")
print("")
print("Watch the console below for agent reasoning...")
print("="*60)
print("")

# Execute the crew
import os
os.makedirs("reports", exist_ok=True)

# Notebooks already run an event loop, so we use the async kickoff and await it
result = await crew.kickoff_async(inputs={"pitch_deck_text": sample_pitch})

print("")
print("="*60)
print("✅ ANALYSIS COMPLETE! Memo saved to reports/investment_memo.md")
print("="*60)

## Section 12: Review the Investment Memo

### WHAT you're seeing:
A professional investment memo with:
- Executive summary (recommendation: INVEST/PASS/INVESTIGATE?)
- Company facts (verified from web)
- Strengths (✅ verified claims)
- Red flags (⚠️ contradictions with confidence levels)
- Unverified items (❓ needs human review)

### WHY are some things flagged?
- **VERIFIED** = pitch claim + web evidence match
- **CONTRADICTED** = pitch claim conflicts with web evidence (RED FLAG)
- **UNVERIFIED** = no web evidence found (human should follow up)

### OBSERVE
**Every single claim has a source:**
- (Pitch: "Traction section") for pitch deck claims
- (Source: https://...) for web findings

No source = no claim allowed.

### ❓ Q&A
**Q: Which claims would I investigate further?**
A: Start with HIGH confidence red flags (most likely to be true issues).

**Q: How would I verify unverified claims?**
A: Customer reference calls, technical deep dive, financial audit.

In [ ]:
# Display the final memo
from IPython.display import Markdown, display

print(f"Tokens used: {result.token_usage}")
display(Markdown(result.raw))

## Section 13: How to Use This System

### For Your Own Startup Pitch:

```python
# Step 1: Replace sample_pitch with your own pitch
my_pitch = """
# My Startup Name
## Company Overview
...(your pitch deck here)...
"""

# Step 2: Run the crew
result = await crew.kickoff_async(inputs={"pitch_deck_text": my_pitch})

# Step 3: See your memo
print(result.raw)
```

### Save to a File:
```python
with open('reports/investment_memo.md', 'w', encoding='utf-8') as f:
    f.write(result.raw)
```

### Modify Agent Behavior:
```python
# Make Agent 1 focus on only financial metrics
claims_analyst.goal = "Extract ONLY financial claims (revenue, growth, CAC, LTV)"

# Re-run the crew
result = await crew.kickoff_async(inputs={"pitch_deck_text": sample_pitch})
```

## 🎓 Learning Summary

### What You Built:
A 4-agent AI system that automates startup investment analysis.

### Key Concepts You Learned:

✅ **Agent Design** = Role + Goal + Backstory + Tools
- Each agent has a specific job and personality
- Backstory helps the model think more effectively
- Tools connect agents to external data (web search)

✅ **Task Definition** = Description + Expected Output + Context
- Tasks tell agents what to do
- Context lets later agents see earlier work
- This is called "context chaining"

✅ **Sequential Orchestration** = Tasks run one after another
- Agent 1's output → Agent 2's input
- Agent 2's output → Agent 3's input
- Critical for workflows where order matters

✅ **Source Attribution** = Every claim must have a source
- Prevents hallucination (AI making up facts)
- Makes memos auditable
- Trust increases when you see sources

✅ **Real-World Application** = Saves 4-8 hours per analysis
- Automates grunt work (research, organizing)
- Lets humans focus on judgment calls
- Scales to 1000+ startups per year

### Why This Matters:
This pattern applies beyond startup analysis:
- **Legal review** = analyze contract → research regulations → flag risks
- **Security audits** = scan code → search vulnerabilities → rank by risk
- **Competitor analysis** = extract claims → research competitors → synthesize SWOT
- **HR screening** = extract CV facts → check references → make recommendation

### Next Steps:
1. Try modifying an agent's goal
2. Add a 5th agent (e.g., Patent analyzer)
3. Test on real pitch decks
4. Deploy as an API for your team

## 🙋 Frequently Asked Questions

### General Questions

**Q: How much does this cost to run?**
A: $0 to start. The Gemini free tier (Google AI Studio key) and Tavily's free plan (1,000 searches a month) cover it; both have rate limits.

**Q: Can I use a different AI model?**
A: Yes! Change the `model=` in `LLM(...)` to:
- "gemini/gemini-2.5-flash" (bigger, slower)
- "gemini/gemini-3.1-flash-lite" (another free daily quota)
- "anthropic/claude-sonnet-5" or "openai/..." (paid keys; install the matching crewai extra)

**Q: What if Tavily can't find information?**
A: Agent 2 says "No data found". Agent 3 marks claims as "Unverified".
This is correct behavior — no fake data!

### Agent Questions

**Q: Why don't Agents 1, 3, 4 have tools?**
A: They don't need them:
- Agent 1: Just reads text (no web needed)
- Agent 3: Just analyzes (no web needed)
- Agent 4: Just writes (no web needed)

Only Agent 2 needs web access.

**Q: Can agents delegate to other agents?**
A: We set `allow_delegation=False`. This keeps control clear.
If True, Agent 1 could ask Agent 2 to do its job (confusing!).

**Q: Why don't we set a temperature?**
A: We leave it at the model's default.
Consistency comes from clear instructions instead: strict roles, "cite every claim", and
"say no data found rather than guess".

### Context Chaining Questions

**Q: What exactly does context=[task1] do?**
A: Agent 2 sees:
- The original pitch
- Agent 1's extracted claims
- Agent 1's reasoning

So Agent 2 knows WHICH claims to research!

**Q: What if Agent 1 gets something wrong?**
A: Agent 3 will catch it when verifying.
Example:
- Agent 1 misses a claim → Agent 2 doesn't research it
- But web data contradicts the pitch anyway
- Agent 3 finds it in research → flags as unverified

### Output Questions

**Q: Why must every claim have a source?**
A: Prevents hallucination.
Example (bad): "Company has 100 employees"
Example (good): "Company has 100 employees (Pitch: Company Overview)"

Without source = we can't verify = don't trust it.

**Q: What does "HIGH confidence" mean?**
A: Strong evidence that the red flag is real.
Example: "Only competitor" but we found 5 competitors → HIGH confidence contradiction.

### Troubleshooting

**Q: Agent 2 is taking forever. Is it stuck?**
A: Probably just searching. Wait 2-3 minutes.
If >5 minutes, there might be an API issue.

**Q: I got an API error. What do I do?**
A: Check your keys in .env
Then re-run the cell. That's it.

**Q: The memo looks weird. Some claims missing?**
A: Agent 4 might have hit token limit (2048 max).
Try a shorter pitch deck.

**Q: Can I run this without internet?**
A: No. Agent 2 needs web access (Tavily).
Agents 1, 3, 4 would work locally, but Agent 2 won't.

## 🚀 Challenge Questions (For You to Think About)

### Easy
1. What would happen if you removed Agent 2? Would the system still work?
   *Hint: Yes, but Agent 3's verification would fail.*

2. Why is Agent 2 slower than the others?
   *Hint: It makes real web API calls.*

3. If Agent 1 misses a claim, which agent might still catch it?
   *Hint: Think about who searches the web.*

### Medium
4. How would you modify the system to ONLY flag high-confidence red flags?
   *Hint: Change Agent 4's goal.*

5. What would you add to Agent 2's goal to also search for founder lawsuits?
   *Hint: Just add it to the description.*

6. If you wanted to analyze 100 startups, how would you batch them?
   *Hint: Loop through them, await crew.kickoff_async() for each one.*

### Hard
7. How would you add a 5th agent (Patent Analyzer) that searches for patents?
   *Hint: Create new agent, new task, add to crew.*

8. The memo sometimes contradicts itself. Why might that happen?
   *Hint: Think about inconsistent web data or LLM reasoning.*

9. How would you make the system more deterministic (same output every time)?
   *Hint: Tighten the expected_output into a fixed template, or ask for structured (JSON) output.*